# Assignment 05: GAT from Scratch (100 points)

Implement Graph Attention Networks from scratch using only PyTorch. Learn how attention coefficients dynamically weight neighbor contributions.

**Topics:** Attention coefficients, LeakyReLU, softmax, multi-head attention, masked attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

---

**WARNING:** Do not modify any cell that you are not explicitly asked to modify. Do not add or remove cells.

## Part 1: Attention Coefficients (20 points)

Given transformed node features $Z = WH$ and an attention vector $\vec{a}$, compute the raw attention scores:

$$e_{ij} = \text{LeakyReLU}\left(\vec{a}_1^T z_i + \vec{a}_2^T z_j\right)$$

Return the full $N \times N$ matrix of attention scores.

In [ ]:
def compute_attention_scores(
    Z: torch.FloatTensor,
    a: torch.FloatTensor,
    negative_slope: float = 0.2
) -> torch.FloatTensor:
    """
    Compute raw attention scores for all node pairs.

    Args:
        Z: Transformed features of shape (N, F')
        a: Attention vector of shape (2*F',)
           First F' entries are a_1, last F' are a_2
        negative_slope: LeakyReLU negative slope

    Returns:
        e: Raw attention scores of shape (N, N)
           e[i,j] = LeakyReLU(a_1^T z_i + a_2^T z_j)
    """
    ### WRITE YOUR SOLUTION HERE ###


    """ END OF THIS PART """

## Part 2: Masked Softmax (15 points)

Apply softmax only over actual neighbors. Non-neighbors should have zero attention.

Given raw scores $e$ and adjacency matrix $A$ (with self-loops), mask out non-neighbors with $-\infty$ before softmax.

$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

In [ ]:
def masked_softmax(
    e: torch.FloatTensor,
    adj: torch.FloatTensor
) -> torch.FloatTensor:
    """
    Apply softmax over neighbors only, masking non-neighbors.

    Args:
        e: Raw attention scores of shape (N, N)
        adj: Adjacency matrix with self-loops of shape (N, N)
             adj[i,j] = 1 if j is a neighbor of i (or i == j)

    Returns:
        alpha: Normalized attention coefficients of shape (N, N)
               alpha[i,j] = 0 if adj[i,j] = 0
               Row i sums to 1 over neighbors
    """
    ### WRITE YOUR SOLUTION HERE ###


    """ END OF THIS PART """

## Part 3: Single-Head GAT Layer (20 points)

Combine the pieces into a single GAT attention head as an `nn.Module`:

1. Linear transform: $Z = HW$
2. Attention scores: $e_{ij} = \text{LeakyReLU}(\vec{a}_1^T z_i + \vec{a}_2^T z_j)$
3. Masked softmax: $\alpha_{ij}$
4. Weighted aggregation: $h_i' = \sum_j \alpha_{ij} z_j$

In [ ]:
class GATHead(nn.Module):
    """
    A single GAT attention head.

    Args:
        in_features: Input feature dimension F
        out_features: Output feature dimension F'
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

    def forward(
        self, H: torch.FloatTensor, adj: torch.FloatTensor
    ) -> tuple[torch.FloatTensor, torch.FloatTensor]:
        """
        Forward pass.

        Args:
            H: Input features of shape (N, F)
            adj: Adjacency matrix with self-loops of shape (N, N)

        Returns:
            H_out: Output features of shape (N, F')
            alpha: Attention coefficients of shape (N, N)
        """
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 4: Multi-Head GAT Layer (25 points)

Implement a multi-head GAT layer that:
- Runs $K$ independent attention heads
- **Concatenates** outputs for intermediate layers: $h_i' = \|_{k=1}^K h_i^{(k)}$
- **Averages** outputs for the final layer: $h_i' = \frac{1}{K}\sum_k h_i^{(k)}$

In [ ]:
class MultiHeadGATLayer(nn.Module):
    """
    Multi-head GAT layer.

    Args:
        in_features: Input dimension F
        out_features: Output dimension per head F'
        num_heads: Number of attention heads K
        concat: If True, concatenate heads (output dim = K*F').
                If False, average heads (output dim = F').
    """

    def __init__(self, in_features: int, out_features: int, num_heads: int, concat: bool = True):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

    def forward(self, H: torch.FloatTensor, adj: torch.FloatTensor) -> torch.FloatTensor:
        """
        Forward pass.

        Args:
            H: Input features of shape (N, F)
            adj: Adjacency with self-loops (N, N)

        Returns:
            H_out: shape (N, K*F') if concat else (N, F')
        """
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 5: Full GAT Model (20 points)

Build a 2-layer GAT for node classification:

- Layer 1: Multi-head with $K = 8$ heads, $F' = 8$ per head, **concatenate** -> output dim 64. Apply ELU and dropout.
- Layer 2: Multi-head with $K = 1$ head, $F' = \text{num\_classes}$, **average**.
- Output: log-softmax.

In [ ]:
class GAT(nn.Module):
    """
    2-layer GAT for node classification.

    Args:
        in_features: Input feature dimension
        num_classes: Number of output classes
        hidden_per_head: Hidden dimension per head in layer 1
        num_heads: Number of heads in layer 1
        dropout: Dropout probability
    """

    def __init__(self, in_features: int, num_classes: int, hidden_per_head: int = 8,
                 num_heads: int = 8, dropout: float = 0.6):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

    def forward(self, H: torch.FloatTensor, adj: torch.FloatTensor) -> torch.FloatTensor:
        """
        Forward pass.

        Args:
            H: Node features (N, F)
            adj: Adjacency with self-loops (N, N)

        Returns:
            out: Log-probabilities (N, num_classes)
        """
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """